## Section D - email.csv

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lower, trim, when, regexp_extract

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CERT Analysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.executorEnv.SPARK_LOCAL_IP", "127.0.0.1") \
    .getOrCreate()


In [ ]:
# utils.py contains all helpers
from helpers import code_quality, parse_date, letters_cleaning

In [ ]:
email_raw = spark.read.csv('../data/email.csv', header=True, inferSchema=True)

In [5]:
email_raw.show(3)

+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+
|                  id|               date|   user|     pc|                  to|                  cc|                 bcc|                from| size|attachments|             content|
+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+
|{C0K9-N2TG12WK-37...|01/02/2010 06:50:45|LRG0155|PC-0450|Rafael.H.Mccall@n...|                NULL|Lysandra_Guerrero...|Lysandra_Guerrero...|29918|          0|apple lot mundane...|
|{F5Y9-A8ON18GE-55...|01/02/2010 07:05:31|KAB0942|PC-8686|Ali.Cole.Mclaughl...|Karly.Amity.Bentl...|                NULL|Karly.Amity.Bentl...|22235|          0|again 15 bought a...|
|{I8R9-T8NW12WZ-66...|01/02/2010 07:06:46|KAB0942|PC-8686|Brent.Colorado.Sa...|           

In [6]:
email_raw.count()

2733360

File `email.csv` Contains: email sending/receiving events. <br>
Columns: id, date, user, pc, to, cc, bcc, from, size, attachments, content <br>
Meaning for GNN: user -> recipient edge (especially external),
large attachments or email to multiple recipients = signal

In [7]:
# cleaning
email = email_raw \
    .dropDuplicates() \
    .filter(F.col("user").isNotNull()) \
    .filter(F.col("from").isNotNull())

In [8]:
# date-parsing
email = parse_date(email)

In [9]:
# In CERT internal domains are of format: dtaa.com.
# Let's mark users that sent emails outside of the organization.
email = email.withColumn(
    "external_recipient",
    ~F.col("to").contains("dtaa.com")  
)

In [10]:
email.show()

+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+-------------------+----+-----+---+----+---------------+-----------------+------------------+
|                  id|               date|   user|     pc|                  to|                  cc|                 bcc|                from| size|attachments|             content|          timestamp|year|month|day|hour|day_of_the_week|not_working_hours|external_recipient|
+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+-------------------+----+-----+---+----+---------------+-----------------+------------------+
|{A1W1-S8VO35UB-98...|01/02/2010 10:14:56|DMR0116|PC-2025|TJN1239@optonline...|Xanthus_David@msn...|Celeste_C_Spears@...|Deborah_Rocha@gma...|52070|          0|similar distinc

In [11]:
# amount of people that received the email
email = email.withColumn(
    "receivers_amount",
    F.size(F.split(F.col("to"), ";"))
)

In [12]:
email.show(3)

+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+-------------------+----+-----+---+----+---------------+-----------------+------------------+----------------+
|                  id|               date|   user|     pc|                  to|                  cc|                 bcc|                from| size|attachments|             content|          timestamp|year|month|day|hour|day_of_the_week|not_working_hours|external_recipient|receivers_amount|
+--------------------+-------------------+-------+-------+--------------------+--------------------+--------------------+--------------------+-----+-----------+--------------------+-------------------+----+-----+---+----+---------------+-----------------+------------------+----------------+
|{A1W1-S8VO35UB-98...|01/02/2010 10:14:56|DMR0116|PC-2025|TJN1239@optonline...|Xanthus_David@msn...|Celeste_C_Spears@...|Deb

In [13]:
print("Top 10 users that sent the largest amount of emails:")
email.groupBy("user") \
     .count() \
     .orderBy(F.desc("count")) \
     .show(10)

Top 10 users that sent the largest amount of emails:
+-------+-----+
|   user|count|
+-------+-----+
|IRG0001|12096|
|ECM0004|11791|
|KNM0008|11550|
|GBM0007|11544|
|IEV0003| 9148|
|TBW0006| 9064|
|CCS0005| 9029|
|LKH0051| 8525|
|SAT0503| 7105|
|CPM0326| 6802|
+-------+-----+
only showing top 10 rows


In [14]:
# internal vs extrernal emails
print("Internal / External Emails:")
email.groupBy("external_recipient") \
     .count() \
     .show()

Internal / External Emails:
+------------------+-------+
|external_recipient|  count|
+------------------+-------+
|              true|1064859|
|             false|1668501|
+------------------+-------+



In [15]:
print("Email's size stats:")
email.select(F.col("size").cast("double").alias("size")) \
     .filter(F.col("size").isNotNull()) \
     .summary("min", "25%", "50%", "75%", "max", "mean") \
     .show()

Email's size stats:
+-------+------------------+
|summary|              size|
+-------+------------------+
|    min|            5774.0|
|    25%|           22875.0|
|    50%|           28466.0|
|    75%|           35441.0|
|    max|          136670.0|
|   mean|30008.620112608656|
+-------+------------------+



In [13]:
email_features = email \
    .groupBy("user", "year", "month", "day") \
    .agg(
        F.count("*").alias("email amounts"),
        F.sum(F.col("external_recipient").cast("int")).alias("external_emails"),
        F.max("receivers_amount").alias("max_receivers"),
        F.avg("receivers_amount").alias("avg_receivers"),
        F.sum(F.col("attachments").cast("int")).alias("attachments_amount"),
        F.sum(F.col("not_working_hours").cast("int")).alias("nightly_emails")
    )

In [14]:
email_features.show(3)

+-------+----+-----+---+-------------+---------------+-------------+------------------+------------------+--------------+
|   user|year|month|day|email amounts|external_emails|max_receivers|     avg_receivers|attachments_amount|nightly_emails|
+-------+----+-----+---+-------------+---------------+-------------+------------------+------------------+--------------+
|DWO0241|2010|    1|  4|            9|              1|            5|1.8888888888888888|                 2|             7|
|LPR0099|2010|    1| 13|            8|              4|            3|             1.625|                 2|             0|
|CCW0209|2010|    1| 19|           14|              4|            4|1.7142857142857142|                 5|             0|
+-------+----+-----+---+-------------+---------------+-------------+------------------+------------------+--------------+
only showing top 3 rows


In [15]:
import pandas as pd
import os

In [17]:
output_dir = "C:/Users/dagak/test_data/email_features_chunks/"


chunks = []
chunk_size = 10000
file_idx = 0

for i, row in enumerate(email_features.toLocalIterator()):
    chunks.append(row.asDict())
    
    if (i + 1) % chunk_size == 0:
        df_chunk = pd.DataFrame(chunks)
        df_chunk.to_parquet(
            f"{output_dir}/email_features_part_{file_idx}.parquet",
            index=False
        )
        print(f"Saved chunk {file_idx}")
        
        chunks = []
        file_idx += 1

# last chunk
if chunks:
    df_chunk = pd.DataFrame(chunks)
    df_chunk.to_parquet(
        f"{output_dir}/email_features_part_{file_idx}.parquet",
        index=False
    )
    print(f"Saved final chunk {file_idx}")

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
def save_email_clean_chunks(email_df, path, chunk_size=10000):
      
    chunks = []
    
    for i, row in enumerate(email_df.toLocalIterator()):
        chunks.append(row.asDict())
        
        if (i + 1) % chunk_size == 0:
            pd.DataFrame(chunks).to_parquet(
                f"{path}/email_clean_part_{i}.parquet",
                index=False
            )
            chunks = []
            print(f"saved {i+1} recs")
    
    
    if chunks:
        pd.DataFrame(chunks).to_parquet(
            f"{path}/email_clean_last.parquet",
            index=False
        )
    
    print("email clean saved by chunks")

In [ ]:
save_email_clean_chunks(
    email,
    "your_link",
    chunk_size=10000
)

saved 10000 recs
saved 20000 recs
saved 30000 recs
saved 40000 recs
saved 50000 recs
saved 60000 recs
saved 70000 recs
saved 80000 recs
saved 90000 recs
saved 100000 recs
saved 110000 recs
saved 120000 recs
saved 130000 recs
saved 140000 recs
saved 150000 recs
saved 160000 recs
saved 170000 recs
saved 180000 recs
saved 190000 recs
saved 200000 recs
saved 210000 recs
saved 220000 recs
saved 230000 recs
saved 240000 recs
saved 250000 recs
saved 260000 recs
saved 270000 recs
saved 280000 recs
saved 290000 recs
saved 300000 recs
saved 310000 recs
saved 320000 recs
saved 330000 recs
saved 340000 recs
saved 350000 recs
saved 360000 recs
saved 370000 recs
saved 380000 recs
saved 390000 recs
saved 400000 recs
saved 410000 recs
saved 420000 recs
saved 430000 recs
saved 440000 recs
saved 450000 recs
saved 460000 recs
saved 470000 recs
saved 480000 recs
saved 490000 recs
saved 500000 recs
saved 510000 recs
saved 520000 recs
saved 530000 recs
saved 540000 recs
saved 550000 recs
saved 560000 recs
s